In [2]:
# Celda 1: Importar librerías
import duckdb
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment


In [3]:
# Celda 2: Crear conexión y esquemas
con = duckdb.connect(database=':memory:')

con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")
con.execute("CREATE SCHEMA IF NOT EXISTS silver;")
con.execute("CREATE SCHEMA IF NOT EXISTS gold;")

In [4]:
# Celda 3: Cargar datos crudos en bronze (corregida para usar barras normales en la ruta)
con.execute("""
    CREATE TABLE bronze.formato_351 AS 
    SELECT * FROM read_csv_auto('Datos/Formato_351.csv');
""")
print("Datos crudos cargados en bronze.formato_351")

Datos crudos cargados en bronze.formato_351


In [5]:
# Celda 4: Procesar datos para silver (limpieza y filtrado)
con.execute("""
    CREATE TABLE silver.formato_351 AS 
    SELECT 
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Razon_Social_Emisor",
        "Fecha de Corte",
        CAST(REPLACE(REPLACE("Valor_Mercado_O_Pres_Pesos", '$', ''), ',', '') AS DOUBLE) AS valor_mercado_limpio,
        "Codigo_Moneda",
        "Pais_Emisor"
    FROM bronze.formato_351 
    WHERE "Nombre Patrimonio" IS NOT NULL
        AND "Nemotecnico" IS NOT NULL
        AND "Nemotecnico" != 'N/A'
        AND "Valor_Mercado_O_Pres_Pesos" IS NOT NULL
        AND CAST(REPLACE(REPLACE("Valor_Mercado_O_Pres_Pesos", '$', ''), ',', '') AS DOUBLE) > 0
        AND "Codigo_Moneda" = 'USD';
""")
print("✓ Datos limpios y filtrados cargados en silver.formato_351")

✓ Datos limpios y filtrados cargados en silver.formato_351


In [6]:
# Celda 5: Agregar posiciones en silver (agrupar por entidad, patrimonio, nemotecnico y fecha)
con.execute("""
    CREATE TABLE silver.posiciones AS 
    SELECT 
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Fecha de Corte",
        SUM(valor_mercado_limpio) AS valor_mercado
    FROM silver.formato_351
    GROUP BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte"
    ORDER BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte";
""")
print("✓ Posiciones agregadas en silver.posiciones")

✓ Posiciones agregadas en silver.posiciones


In [7]:
# Celda 6: Calcular pesos en silver
con.execute("""
    CREATE TABLE silver.posiciones_con_pesos AS 
    SELECT 
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Fecha de Corte",
        valor_mercado,
        SUM(valor_mercado) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Fecha de Corte") AS valor_portafolio,
        ROUND(valor_mercado / SUM(valor_mercado) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Fecha de Corte"), 4) AS peso
    FROM silver.posiciones
    ORDER BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte";
""")
print("✓ Pesos calculados en silver.posiciones_con_pesos")

✓ Pesos calculados en silver.posiciones_con_pesos


In [8]:
# Celda 7: Calcular cambios de posiciones en silver
con.execute("""
    CREATE TABLE silver.movimientos AS 
    SELECT 
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Fecha de Corte",
        valor_mercado,
        valor_portafolio,
        peso,
        LAG(peso) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico" ORDER BY "Fecha de Corte") AS peso_anterior,
        ROUND(peso - LAG(peso) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico" ORDER BY "Fecha de Corte"), 4) AS cambio_peso,
        ROUND(valor_mercado - LAG(valor_mercado) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico" ORDER BY "Fecha de Corte"), 2) AS cambio_valor,
        CASE 
            WHEN valor_mercado - LAG(valor_mercado) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico" ORDER BY "Fecha de Corte") > 0 THEN 'COMPRA'
            WHEN valor_mercado - LAG(valor_mercado) OVER (PARTITION BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico" ORDER BY "Fecha de Corte") < 0 THEN 'VENTA'
            ELSE 'SIN_CAMBIO'
        END AS tipo_movimiento
    FROM silver.posiciones_con_pesos
    ORDER BY "Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte";
""")
print("✓ Movimientos calculados en silver.movimientos")

✓ Movimientos calculados en silver.movimientos


In [9]:
# Celda 8: Filtrar movimientos significativos en gold
con.execute("""
    CREATE TABLE gold.movimientos_significativos AS 
    SELECT 
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Fecha de Corte",
        valor_mercado,
        valor_portafolio,
        peso,
        cambio_peso,
        cambio_valor,
        tipo_movimiento
    FROM silver.movimientos
    WHERE ABS(cambio_peso) > 0.01
        AND tipo_movimiento != 'SIN_CAMBIO'
    ORDER BY ABS(cambio_peso) DESC, "Fecha de Corte" DESC;
""")
print("✓ Movimientos significativos en gold.movimientos_significativos")

✓ Movimientos significativos en gold.movimientos_significativos


In [10]:
# Celda 9: Crear reportes por tipo de movimiento en gold
con.execute("""
    CREATE TABLE gold.compras AS 
    SELECT * FROM gold.movimientos_significativos
    WHERE tipo_movimiento = 'COMPRA'
    ORDER BY cambio_valor DESC;
""")

con.execute("""
    CREATE TABLE gold.ventas AS 
    SELECT * FROM gold.movimientos_significativos
    WHERE tipo_movimiento = 'VENTA'
    ORDER BY ABS(cambio_valor) DESC;
""")

print("✓ Reporte de compras en gold.compras")
print("✓ Reporte de ventas en gold.ventas")

✓ Reporte de compras en gold.compras
✓ Reporte de ventas en gold.ventas


In [11]:
# Celda 10: Crear consenso de inversión en gold
con.execute("""
    CREATE TABLE gold.consenso_inversion AS 
    SELECT 
        silver.movimientos."Nemotecnico",
        silver.formato_351."Razon_Social_Emisor",
        silver.movimientos.tipo_movimiento,
        COUNT(DISTINCT silver.formato_351."Nombre Patrimonio") AS cantidad_fondos,
        COUNT(*) AS total_movimientos,
        ROUND(AVG(silver.movimientos.valor_mercado), 2) AS promedio_inversion,
        ROUND(SUM(silver.movimientos.valor_mercado), 2) AS monto_total_invertido
    FROM silver.formato_351
    INNER JOIN silver.movimientos ON 
        silver.formato_351."Nemotecnico" = silver.movimientos."Nemotecnico"
    WHERE silver.movimientos.tipo_movimiento IN ('COMPRA', 'VENTA')
    GROUP BY silver.movimientos."Nemotecnico", silver.formato_351."Razon_Social_Emisor", silver.movimientos.tipo_movimiento
    ORDER BY monto_total_invertido DESC;
""")
print("✓ Consenso de inversión en gold.consenso_inversion")

✓ Consenso de inversión en gold.consenso_inversion


In [12]:
# Celda 11: Verificar resultados
print("="*60)
print("=== VERIFICACIÓN DE ESQUEMAS ===")
print("="*60)

print("\n📊 BRONZE (Datos crudos sin modificar):")
bronze_count = con.execute("SELECT COUNT(*) AS total FROM bronze.formato_351").fetchall()
print(f"  Total registros: {bronze_count[0][0]:,}")

print("\n🔧 SILVER (Datos limpios y procesados):")
silver_count = con.execute("SELECT COUNT(*) AS total FROM silver.movimientos WHERE cambio_peso IS NOT NULL").fetchall()
print(f"  Total movimientos: {silver_count[0][0]:,}")

print("\n🏆 GOLD (Señales de inversión consolidadas):")
gold_compras = con.execute("SELECT COUNT(*) AS total FROM gold.compras").fetchall()
gold_ventas = con.execute("SELECT COUNT(*) AS total FROM gold.ventas").fetchall()
print(f"  Señales de compra: {gold_compras[0][0]:,}")
print(f"  Señales de venta: {gold_ventas[0][0]:,}")

print("\n" + "="*60)
print("✓ Pipeline completado exitosamente")
print("="*60)

=== VERIFICACIÓN DE ESQUEMAS ===

📊 BRONZE (Datos crudos sin modificar):
  Total registros: 2,290,145

🔧 SILVER (Datos limpios y procesados):
  Total movimientos: 6,944

🏆 GOLD (Señales de inversión consolidadas):
  Señales de compra: 751
  Señales de venta: 678

✓ Pipeline completado exitosamente


In [13]:
# Celda 12: Exportar reporte en Excel
# Definir ruta del reporte
ruta_reporte = r"Datos/reporte_duckdb_prueba.xlsx"

# Extraer datos de gold en DataFrames de pandas
movimientos_df = pd.DataFrame(
    con.execute("SELECT * FROM gold.movimientos_significativos").fetchall(),
    columns=["Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte", 
             "Valor_Mercado", "Valor_Portafolio", "Peso", "Cambio_Peso", "Cambio_Valor", "Tipo_Movimiento"]
)

compras_df = pd.DataFrame(
    con.execute("SELECT * FROM gold.compras").fetchall(),
    columns=["Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte", 
             "Valor_Mercado", "Valor_Portafolio", "Peso", "Cambio_Peso", "Cambio_Valor", "Tipo_Movimiento"]
)

ventas_df = pd.DataFrame(
    con.execute("SELECT * FROM gold.ventas").fetchall(),
    columns=["Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico", "Fecha de Corte", 
             "Valor_Mercado", "Valor_Portafolio", "Peso", "Cambio_Peso", "Cambio_Valor", "Tipo_Movimiento"]
)

consenso_df = pd.DataFrame(
    con.execute("SELECT * FROM gold.consenso_inversion").fetchall(),
    columns=["Nemotecnico", "Razon_Social_Emisor", "Tipo_Movimiento", "Cantidad_Fondos", 
             "Total_Movimientos", "Promedio_Inversion", "Monto_Total_Invertido"]
)

# Crear archivo Excel con múltiples hojas
with pd.ExcelWriter(ruta_reporte, engine="xlsxwriter") as writer:
    movimientos_df.to_excel(writer, sheet_name="Movimientos", index=False)
    compras_df.to_excel(writer, sheet_name="Compras", index=False)
    ventas_df.to_excel(writer, sheet_name="Ventas", index=False)
    consenso_df.to_excel(writer, sheet_name="Consenso", index=False)

    workbook = writer.book

    # Definir formatos
    header = workbook.add_format({
        "bold": True,
        "align": "center",
        "valign": "vcenter",
        "border": 1,
        "bg_color": "#4472C4",
        "font_color": "FFFFFF"
    })

    money = workbook.add_format({
        "num_format": "$#,##0.00",
        "border": 1,
        "align": "right"
    })

    percent = workbook.add_format({
        "num_format": "0.00%",
        "border": 1,
        "align": "center"
    })

    text = workbook.add_format({
        "border": 1,
        "align": "left"
    })

    # Función para aplicar formato
    def formato_hoja(nombre, df):
        ws = writer.sheets[nombre]

        # Encabezados
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, header)

        # Ancho de columnas y formato de datos
        ws.set_column("A:A", 25, text)
        ws.set_column("B:B", 35, text)
        ws.set_column("C:C", 15, text)
        ws.set_column("D:D", 15, text)
        ws.set_column("E:E", 18, money)
        ws.set_column("F:F", 18, money)
        ws.set_column("G:G", 12, percent)
        ws.set_column("H:H", 12, percent)
        ws.set_column("I:I", 18, money)
        ws.set_column("J:J", 15, text)

        # Congelar encabezado
        ws.freeze_panes(1, 0)
        
        # Agregar filtros
        ws.autofilter(0, 0, len(df), len(df.columns) - 1)

    # Aplicar formato a hojas principales
    formato_hoja("Movimientos", movimientos_df)
    formato_hoja("Compras", compras_df)
    formato_hoja("Ventas", ventas_df)
    
    # Formato especial para Consenso
    ws_consenso = writer.sheets["Consenso"]
    for i, col in enumerate(consenso_df.columns):
        ws_consenso.write(0, i, col, header)
    
    ws_consenso.set_column("A:A", 15, text)
    ws_consenso.set_column("B:B", 35, text)
    ws_consenso.set_column("C:C", 15, text)
    ws_consenso.set_column("D:D", 15, money)
    ws_consenso.set_column("E:E", 18, money)
    ws_consenso.set_column("F:F", 18, money)
    ws_consenso.set_column("G:G", 20, money)
    ws_consenso.freeze_panes(1, 0)
    ws_consenso.autofilter(0, 0, len(consenso_df), len(consenso_df.columns) - 1)

print("✓ Reporte Excel generado exitosamente")
print(f"  Ubicación: {ruta_reporte}")
print(f"  Hojas: Movimientos, Compras, Ventas, Consenso")

✓ Reporte Excel generado exitosamente
  Ubicación: Datos/reporte_duckdb_prueba.xlsx
  Hojas: Movimientos, Compras, Ventas, Consenso
